# 当前美国期权市场 Underlyings

从 Massive 的 OPRA 期权合约参考接口读取运行日仍有效（`expired=false`）的合约，并按 `underlying_ticker` 汇总。这里的“市场”特指 Massive 覆盖的美国上市期权，不包含加密期权、场外期权或其他国家/地区市场。结果是合约目录覆盖范围，不代表当日有成交或可交易。

In [1]:
from datetime import date, datetime, timezone
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'trading').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

AS_OF = date.today().isoformat()
CACHE = ROOT / 'data' / 'reference' / 'provider=massive' / 'option-underlyings' / f'as_of={AS_OF}' / 'contracts.json'
print({'as_of': AS_OF, 'cache': str(CACHE)})

{'as_of': '2026-07-16', 'cache': '/Users/zhaoqian/Code/Github/trader/data/reference/provider=massive/option-underlyings/as_of=2026-07-16/contracts.json'}


## 1. 获取当前合约目录

首次运行需要在启动 Jupyter 前设置 `MASSIVE_API_KEY`。查询按到期月份拆分，每个成功分片立即缓存；中途失败后重新运行会从未完成分片继续。最后还有一个六年以后的开放尾部分片，因此不会因固定截止日漏掉超长期合约。将 `REFRESH=True` 可强制刷新全部分片。

In [2]:
from trading.adapters.massive import MassiveClient, MassiveConfig

REFRESH = False
if CACHE.exists() and not REFRESH:
    payload = json.loads(CACHE.read_text(encoding='utf-8'))
    contracts = payload['contracts']
    retrieved_at = payload['retrieved_at']
    source = 'cache'
else:
    if not os.environ.get('MASSIVE_API_KEY'):
        raise RuntimeError('请先在 shell 中设置 MASSIVE_API_KEY，再启动 Jupyter。')
    base_config = MassiveConfig.from_env()
    client = MassiveClient(MassiveConfig(
        api_key=base_config.api_key, timeout_seconds=120, max_retries=base_config.max_retries,
    ))

    def next_month(value):
        return date(value.year + value.month // 12, value.month % 12 + 1, 1)

    first_day = date.fromisoformat(AS_OF)
    horizon = date(first_day.year + 6, first_day.month, 1)
    shards, cursor = [], first_day
    while cursor < horizon:
        boundary = next_month(cursor)
        shards.append((cursor, boundary))
        cursor = boundary
    shards.append((cursor, None))  # 开放尾部：覆盖六年以后的合约

    shard_root = CACHE.parent / 'shards'
    shard_root.mkdir(parents=True, exist_ok=True)
    contracts = []
    for index, (start, end) in enumerate(shards, 1):
        label = f'{start.isoformat()}_{end.isoformat() if end else "tail"}'
        shard_path = shard_root / f'{label}.json'
        if shard_path.exists() and not REFRESH:
            rows = json.loads(shard_path.read_text(encoding='utf-8'))
            shard_source = 'cache'
        else:
            params = {
                'as_of': AS_OF, 'expired': False, 'limit': 1000,
                'sort': 'ticker', 'order': 'asc',
                'expiration_date.gte': start.isoformat(),
                'expiration_date.lt': end.isoformat() if end else None,
            }
            rows = []
            for _, page in client.pages('/v3/reference/options/contracts', params):
                page_rows = page.get('results', [])
                if not isinstance(page_rows, list):
                    raise TypeError('Massive options/contracts results 应为 list')
                rows.extend(page_rows)
            shard_path.write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
            shard_source = 'api'
        contracts.extend(rows)
        print(f'[{index}/{len(shards)}] {label}: {len(rows)} contracts ({shard_source})')

    contracts = list({row['ticker']: row for row in contracts}.values())
    retrieved_at = datetime.now(timezone.utc).isoformat()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps({
        'provider': 'massive', 'market': 'OPRA', 'as_of': AS_OF,
        'retrieved_at': retrieved_at, 'contracts': contracts,
    }, ensure_ascii=False), encoding='utf-8')
    source = 'api'

print({'source': source, 'retrieved_at': retrieved_at, 'contracts': len(contracts)})

IncompleteRead: IncompleteRead(205676 bytes read, 24098 more expected)

## 2. Underlying 清单与覆盖统计

In [ ]:
required = {'ticker', 'underlying_ticker', 'contract_type', 'expiration_date'}
missing = required - set().union(*(row.keys() for row in contracts)) if contracts else required
if missing:
    raise ValueError(f'合约结果缺少字段: {sorted(missing)}')

frame = pd.DataFrame(contracts)
frame['expiration_date'] = pd.to_datetime(frame['expiration_date'], errors='coerce')
frame['contract_type'] = frame['contract_type'].str.lower()
underlyings = (frame.groupby('underlying_ticker', as_index=False)
    .agg(contracts=('ticker', 'nunique'),
         calls=('contract_type', lambda s: int((s == 'call').sum())),
         puts=('contract_type', lambda s: int((s == 'put').sum())),
         first_expiry=('expiration_date', 'min'),
         last_expiry=('expiration_date', 'max'))
    .sort_values(['contracts', 'underlying_ticker'], ascending=[False, True])
    .reset_index(drop=True))

display(pd.DataFrame([{
    'as_of': AS_OF, 'retrieved_at_utc': retrieved_at,
    'underlyings': len(underlyings), 'active_contracts': frame['ticker'].nunique(),
    'earliest_expiry': frame['expiration_date'].min(),
    'latest_expiry': frame['expiration_date'].max(),
}]))
display(underlyings)

## 3. 搜索和快速检查

In [ ]:
QUERY = ''  # 例如 SPX、NVDA；留空显示合约数最多的 50 个
matches = underlyings[underlyings['underlying_ticker'].str.contains(QUERY, case=False, regex=False)]
display(matches.head(50))

ax = underlyings.head(25).sort_values('contracts').plot.barh(
    x='underlying_ticker', y='contracts', figsize=(10, 8), legend=False, color='#2774AE'
)
ax.set(title=f'Active option contracts by underlying ({AS_OF})', xlabel='Contracts', ylabel='Underlying')
plt.tight_layout()

## 4. 导出清单

CSV 与 notebook 使用同一查询日期，便于后续研究直接复用。

In [ ]:
OUTPUT = CACHE.with_name('underlyings.csv')
underlyings.to_csv(OUTPUT, index=False)
print(OUTPUT)